分词器（tokenizer）常被视为大语言模型（LLM）的一部分，但实际上它有着相对独立的设计与训练流程。
分词算法不是唯一的：不同的分词算法（如BPE、WordPiece）会出现不同的划分结果；
分词器会影响模型能力：token划分得太“碎片化”或太“笼统”，都会影响表达效率和泛化能力；
它是独立优化的模块：通常先在大规模文本上训练词表（vocab），再固定下来供模型使用。
# 训练分词器
训练一个用于现代大型语言模型的分词器可以拆成四步：准备语料 → 初始化基础单元(不同的分词算法，可省略) → 统计并迭代合并 → 输出产物并用于编码、解码。


1 准备语料
多语言或混合语料场景中，应统计各语言占比，并评估是否对低资源语言进行过采样或定向保留，避免词表被高频语言主导。否则，语料类型与语言分布不均衡会加剧低资源语言的token碎片化，可能会降低其任务性能
语言	语料量
中文	200 GB
英文	150 GB
法语	10 GB
韩文	5 GB
这是一个典型的多语言语料不平衡场景。若将上述语料不经处理直接混合训练分词器，其统计过程会被中文和英文主导，导致法语与韩文的常见字串在合并阶段难以进入高频统计，从而无法占据足够的词表空间，最终在vocab中会出现大量被切得过碎的token，形成严重碎片化，下游LLM在法语与韩文任务上会因此表现显著劣化。
因此，在准备语料时，需要先按语言统计语料占比，并根据目标能力设定合理的采样策略。例如将语料比例调整为中文:英文:法语:韩文=4:4:1:1或者采用完全均衡策略。通过对高资源语言下采样或对低资源语言过采样、增强，可以获得更符合目标分布的训练语料。
Step2  对原始文本进行清洗和标准化是必须的步骤，包含去除或屏蔽无关元数据、修正或删除乱码与非法字符、统一字符编码，以及对带有敏感信息或隐私的语料要提前进行脱敏处理与合规检查，明确哪些信息不可用于训练并记录数据来源与许可。
 值得注意的是，数据脱敏不仅是出于隐私保护与合规要求，同时也有助于提升下游文本建模与分词过程的稳定性
这样可以有效降低语料中无效的多样性，使分词器更专注于建模具有统计规律的语言模式，从而提升词表的利用效率、一致性。从香农信息论的角度来看，数据脱敏可以视为一种结构化的去噪过程——通过压缩或减少高熵但低语义价值的信号（如具体身份信息），提高语料中有效信号的占比，可以协助LLM在后面的训练过程更倾向于学习可复用的语义结构，而非记忆偶然出现的实例细节。
Step3  建议保留一小部分未参与训练的验证语料比如训练集:验证集＝99:1，用来在训练过程中评估分词器对真实文本的编码效率与平均token长度等统计指标。


In [1]:
import re
from typing import List, Callable
from transformers import pipeline

In [2]:
"""
"sentiment-analysis": 情感分析（判断一句话是褒义还是贬义）。
"text-generation": 文本生成（比如 GPT 写的续写）。
"translation_zh_to_en": 翻译（中译英）。
"question-answering": 问答系统（从文章中找答案）。
"summarization": 自动摘要
ner: Named Entity Recognition (NER) 定义“整个工厂的任务类型”。
找到模型：下载 BERT 模型的权重（大脑）。
自动找到对应的分词器：它会去云端寻找专门为这个模型设计的 Tokenizer。
注意： 因为 NER 任务对分词的要求非常严格（必须保留原始字符的位置，不能乱切），所以这个 Pipeline 会自动加载那个模型的 专属分词器，确保切分后的 Token 能对齐到正确的人名或地名上。
ner tell model 把分词器、BERT模型和后处理逻辑全部串联起来，目标是识别实体。”

"""

ner_pipeline = pipeline(
    "ner",
    model="ckiplab/bert-base-chinese-ner",
    aggregation_strategy="simple"
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/407M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/407M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: ckiplab/bert-base-chinese-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [27]:
def ner_mask(text: str) -> str:
  """
  entities = [
    {"entity_group": "PER", "start": 0, "end": 2, "word": "小明", "score": 0.99},
    {"entity_group": "LOC", "start": 15, "end": 17, "word": "北京", "score": 0.98}
  ]
  """
  entities = ner_pipeline(text)
  print(entities)
  spans = []

  for ent in entities:
    label = ent["entity_group"]
    start = ent["start"]
    end = ent["end"]

    if label == "PER":
      spans.append((start, end, "[NAME]"))
    elif label == "LOC":
      spans.append((start, end, "[PLACE]"))
  """
  (0, 6, "[PLACE]") — 长的排第一
  (0, 2, "[PLACE]") — 短的排第二
  """
  spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))

  """
  在 NLP 中，模型有时会识别出互相重叠的词（例如同时识别出“张三”和“张三丰”）。如果不处理，程序可能会尝试在同一个位置替换两次，导致结果变成 [NAME]丰 或者更糟糕的乱码。
  模型识别出了两个 spans（排序后）：
  (0, 3, "[NAME]") — "张三丰" (长)
  (0, 2, "[NAME]") — "张三" (短)
  (10, 23, "[NAME]") — "beijing"
  """
  filtered_span = []
  last_end = -1
  for start,  end, tag in spans:
    if start >= last_end:
      filtered_span.append((start, end, tag))
      last_end = end

  result = []
  last_idx = 0
  for start, end, tag in filtered_span:
    result.append(text[last_idx:start])
    result.append(tag)
    last_idx = end
  result.append(text[last_idx:])
  return "".join(result)



In [33]:
a = ner_mask("我叫小明。")
a

[{'entity_group': 'PERSON', 'score': np.float32(0.977589), 'word': '小', 'start': 2, 'end': 3}, {'entity_group': 'PERSON', 'score': np.float32(0.9970432), 'word': '明', 'start': 3, 'end': 4}]


'我叫小明。'

In [29]:
class DesensitizationPipeline:
  def __init__(self):
    """
    Callable[[str], str]:
    Callable 表示“可调用对象”，通常指函数。
    [[str], str] 进一步规定了这个函数的“规格”：它必须接收一个字符串 (str) 作为输入，并返回一个字符串 (str) 作为输出。
    """
    self.steps: List[Callable[[str], str]] = []

  def add_step(self, func: Callable[[str], str]):
    self.steps.append(func)

  def run(self, text:str) ->str:
    for step in self.steps:
      text = step(text)
    return text



In [30]:
def normalize_text(text: str) -> str:
  return text.strip()

def mask_phone(text: str) -> str:
  return re.sub(r'1[3-9]\d{9}', '[PHONE]', text)

def mask_email(text: str) -> str:
  return re.sub(r'[a-zA-Z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '[EMAIL]', text)

def mask_address(text: str) -> str:
    return re.sub(
        r'(居住于|现居住于|现居于|地址)([\u4e00-\u9fa5A-Za-z0-9]+)',
        r'\1[PLACE]',
        text
    )
"""
(?<=^): 肯定逆向环视（Positive Lookbehind）。要求当前位置的前面必须是字符串的开头。
(?<=[，。！？]): 要求当前位置的前面必须是中文标点符号。
(?: ... ): 非捕获分组。只是为了把上面两个逻辑“或”起来，不保存匹配结果。
"""
def mask_name(text: str) -> str:
    return re.sub(
        r'(?:(?<=^)|(?<=[，。！？]))([\u4e00-\u9fa5]{2,3})(的)',
        r'[NAME]\2',
        text
    )

def clean_punctuation(text: str) -> str:
  return text

def build_pipeline():
    """
    组装流水线：建议遵循“预处理 -> 高准确率正则 -> AI识别 -> 低准确率正则兜底”的顺序
    """
    p = DesensitizationPipeline()

    p.add_step(normalize_text)
    p.add_step(mask_phone)
    p.add_step(mask_email)
    p.add_step(ner_mask)
    p.add_step(mask_address)
    p.add_step(mask_name)
    p.add_step(clean_punctuation)

    return p

In [32]:
test_text = "小明的邮箱是test@gmail.com，电话是13312311111，现在居住于重庆两江新区的xxx小区。"
ds_pipeline = build_pipeline()
print("origin->", test_text)
print("after->", ds_pipeline.run(test_text))


origin-> 小明的邮箱是test@gmail.com，电话是13312311111，现在居住于重庆两江新区的xxx小区。
[{'entity_group': 'PERSON', 'score': np.float32(0.948194), 'word': '小', 'start': 0, 'end': 1}, {'entity_group': 'PERSON', 'score': np.float32(0.99784803), 'word': '明', 'start': 1, 'end': 2}, {'entity_group': 'GPE', 'score': np.float32(0.9965557), 'word': '重', 'start': 30, 'end': 31}, {'entity_group': 'GPE', 'score': np.float32(0.9980154), 'word': '庆', 'start': 31, 'end': 32}, {'entity_group': 'CARDINAL', 'score': np.float32(0.48744228), 'word': '两', 'start': 32, 'end': 33}]
after-> [NAME]的邮箱是[EMAIL]，电话是[PHONE]，现在居住于[PLACE]。


2 预分词阶段
## Step1
预分词的主要任务是将原始文本切分成可统计、可合并的基础单元，例如字符、字节或Unicode片段。常见策略包括基于空格和标点的切分、按Unicode类别划分，或直接采用字节级切分。需要注意的是并不是所有的分词器都需要用户显式进行预分词。例如基于SentencePiece的分词器将标准化和预分词逻辑内置，因此无需在外部额外执行预分词步骤。



In [40]:
"""
基于空格和标点的切分策略：
一个完整的句子中遇到空格或者标点（.,!?[]{}...）
可以分为独立的tokens，该方法适用于大多数预分词处理过程。
"""
def part(text):
  text = re.sub(r'([.,!?:;()"\'\[\]{}])',r'\1', text)
  # text = re.sub(r'([.,!?;:()"\'\[\]{}])', '', text)

  tokens = text.split()
  return tokens

s = "I , ,like ! NLP"
print(part(s))

['I', ',', ',like', '!', 'NLP']


In [52]:
import unicodedata
def get_char_category(ch:str) -> str:
  # 获取Unicode标准定义的分类（如'Lu'代表大写字母,'Po'代表其它标点）
  cat = unicodedata.category(ch)
  # print(cat, ch)

  # 判定是否为中文字符（常用基本汉字区间）
  if "\u4e00" <= ch <= '\u9fff':
    return "CJK"

  if ch.isalpha():
    return "ALPHA"

  if cat.startswith("P"):
    return "PUNCT"
  # emojo, space or ...
  return "OTHER"

def segment_by_unicode_category(text:str):
  if not text:
    return []
  segments = []
  buffer = [text[0]]

  prev_type = get_char_category(text[0])

  for ch in text[1:]:
    # print("ch->", ch)
    curr_type = get_char_category(ch)

    if curr_type == prev_type:
      buffer.append(ch)
      #print("b->", buffer)
    else:
      segments.append(("".join(buffer), prev_type))
      buffer = [ch]
      prev_type = curr_type
    #print("s->",segments)


  segments.append(("".join(buffer), prev_type))
  #print("s->",segments)

  tokens = [seg for seg, _ in segments]
  return tokens

s = "Hello👋👋，Datawhale成立于2018年！！！"
result = segment_by_unicode_category(s)
print("原始文本:", s)
print("分段结果:", result)


Lu H
Ll e
Ll l
Ll l
Ll o
So 👋
So 👋
Po ，
Lu D
Ll a
Ll t
Ll a
Ll w
Ll h
Ll a
Ll l
Ll e
Lo 成
Lo 立
Lo 于
Nd 2
Nd 0
Nd 1
Nd 8
Lo 年
Po ！
Po ！
Po ！
原始文本: Hello👋👋，Datawhale成立于2018年！！！
分段结果: ['Hello', '👋👋', '，', 'Datawhale', '成立于', '2018', '年', '！！！']


nicode就像给全世界所有字符发的“身份证号”，不管是英文A、汉字“中”、还是emoji 😄等不同类型的字符都在Unicode里有一个唯一编号比如A是U+0041，“中”是U+4E2D。但“身份证号”本身只是一个抽象编号，电脑不能直接存储。

  UTF-8就像把这个字符对应的“身份证号”写进电脑的具体方式。它规定这个字符的编号应该用几个字节、按什么规则写下来。英文常用字符在UTF-8中只需要1个字节，而中文通常需要3个字节。

In [61]:
def tokenize_byte_level(text):
  tokens = []
  for ch in text:
    utf8_bytes = ch.encode('utf-8')
    # print(utf8_bytes)
    hex_bytes = [f"{b:02X}" for b in utf8_bytes]
    #print(f"{ch} convert to utf8: {hex_bytes}")
    tokens.append(hex_bytes)
  return tokens

s = "ALL is NLP"
print(tokenize_byte_level(s))

A convert to utf8: ['41']
L convert to utf8: ['4C']
L convert to utf8: ['4C']
  convert to utf8: ['20']
i convert to utf8: ['69']
s convert to utf8: ['73']
  convert to utf8: ['20']
N convert to utf8: ['4E']
L convert to utf8: ['4C']
P convert to utf8: ['50']
[['41'], ['4C'], ['4C'], ['20'], ['69'], ['73'], ['20'], ['4E'], ['4C'], ['50']]


在LLM的token划分中，常见策略包括：

（1）基于规则的预分词（如按空格和标点切分）；

（2）按Unicode类别分段（如连续汉字、连续拉丁字母或数字）；

（3）更底层的UTF-8字节级切分。

UTF-8字节级策略具有最强的通用性，它把任意文本统一拆为字节序列，从而从根本上减少未词汇表外（OOV）问题并覆盖任意字符集。但因为它以最细粒度开始，训练时通常需要更多轮的共现统计与合并来把零散字节压缩为紧凑且具语义的token，才能在Transformer计算效率与语义表征之间取得平衡。

当token划分得越碎片化（即总token数量越多），注意力机制的效率就越低。

##Step2  对大多数以空格为词边界的语言，可先用正则表达式按单词边界和标点进行初步划分，而对中文、日文等不以空格为词界的语言则通常采用逐字符或基于字的初始单元来保证覆盖性。

##Step3  预分词生成的基础单元序列将作为后续统计合并的输入，务必保存该序列与对应位置信息以便在训练过程反复高效更新。



In [62]:
# 预分词保存对应位置信息的实现示例
def btp_hex_list(text):
  tokens = []
  t = []

  for idx, char in enumerate(text):
    utf8_bytes = char.encode("utf-8")
    hex_bytes = " ".join(f"{b:02X}" for b in utf8_bytes)
    tokens.append(
        {
            'char': char,
            'bytes': hex_bytes,
            'idx': idx,
            "end": idx + 1
        }
    )
    t.extend([f"{b:02X}" for b in utf8_bytes])

  return tokens, t

text = "Hi，你好🐋"
tokens, t = btp_hex_list(text)
for i in tokens:
    print(i)
print(t)

{'char': 'H', 'bytes': '48', 'idx': 0, 'end': 1}
{'char': 'i', 'bytes': '69', 'idx': 1, 'end': 2}
{'char': '，', 'bytes': 'EF BC 8C', 'idx': 2, 'end': 3}
{'char': '你', 'bytes': 'E4 BD A0', 'idx': 3, 'end': 4}
{'char': '好', 'bytes': 'E5 A5 BD', 'idx': 4, 'end': 5}
{'char': '🐋', 'bytes': 'F0 9F 90 8B', 'idx': 5, 'end': 6}
['48', '69', 'EF', 'BC', '8C', 'E4', 'BD', 'A0', 'E5', 'A5', 'BD', 'F0', '9F', '90', '8B']


3 统计并迭代更新
Step1  子词候选统计

BPE：统计当前字符、子词序列中相邻对的出现频次，每次贪心合并出现频次最高的相邻对，迭代构建词表——其决策仅基于频率统计。
WordPiece：评估合并或保留某些子词组合对语料似然即语言模型性能的贡献，选择能显著提升语料拟合度的合并操作。
Unigram：从一个过大的种子词表出发，初始化每个token的概率。
SentencePiece：是一个语言无关的子词分词框架，提供统一的训练与编码流程，支持多种分词算法（如BPE和Unigram）。这些算法在同一框架下独立使用，而非直接融合使用，其互补性体现在不同任务和数据条件下的适用性差异。

BPE (Byte Pair Encoding，字节对编码) 是目前大语言模型（如 GPT 系列、Llama）最主流的分词算法。它的核心思想是：从字符开始，不断合并出现频率最高的相邻字符对，直到达到预设的词表大小。
你可以把它理解为一个“拼积木”的过程：从最小的零件开始，把经常在一起出现的零件粘成一个新组件。
------------------------------
## 1. BPE 的核心工作流程
假设我们有一段简单的语料："hug", "pug", "pun", "bun"。
## 第一步：初始化（最小单位）
将所有词拆解为单个字符，并统计词频：

* h u g: 10次
* p u g: 5次
* p u n: 12次
* b u n: 4次
此时，我们的词表（Vocab）只有：h, u, g, p, n, b。

## 第二步：统计并合并最频繁的相邻对

   1. 第一次迭代：我们发现 u 和 n 经常在一起出现（12+4=16次）。
   * 动作：将 u 和 n 合并为新 Token un。
      * 更新词表：h, u, g, p, n, b, un。
   2. 第二次迭代：我们发现 p 和 un 经常在一起出现（12次）。
   * 动作：合并为 pun。
      * 更新词表：..., un, pun。
   
## 第三步：重复直到满足条件
这个过程会一直持续，直到合并出的新 Token 数量达到了你设定的词表大小（例如 50,000）。
------------------------------
## 2. BPE 算法的三个天才之处

   1. 完美解决“生僻字” (Out-of-Vocabulary)：
   即使遇到一个从未见过的长单词，BPE 也能把它切成它认识的“子词（Subwords）”。最差的情况也就是切成一个个字母，绝对不会出现无法识别的 [UNK] 符号。
   2. 效率与语义的平衡：
   * 高频词：像 "the", "ing" 会被合并成一个 Token（节省空间，提升速度）。
      * 低频词：会被切碎成几个 Token（模型依然能通过子词理解含义）。
   3. 字节级 BPE (Byte-level BPE)：
   现代模型（如 GPT-2 以后）不直接处理字符，而是处理 Bytes（字节）。正如你之前写的 02X 16进制代码那样。这让一个分词器就能处理全球所有语言（中文、英文、日文、甚至表情包），因为所有文字在底层都是字节。

------------------------------
## 3. 一个直观的例子
如果你训练了一个 BPE 分词器：

* 常用词："transformer" $\rightarrow$ [transformer] (1个 Token)
* 罕见词："transformership" $\rightarrow$ [transformer, ship] (2个 Token)

## 总结
BPE 就是通过统计学的方法，自动找出语言中最有意义的“片段”，并将它们固定在词表里。

In [69]:
# 字符分词器
class CharacterTokenizer:
  def __init__(self) -> None:
    pass

  def encode(self, text):
    # 返回它在 Unicode 标准中对应的整数编号（Code Point）。
    return [ord(ch) for ch in text]

  def decode(self, indices):
    # chr 是 character（字符）的缩写。
    return "".join([chr(i) for i in indices])


tokenizer = CharacterTokenizer()
string = "hi，很好的，terrific！🐋"
indices = tokenizer.encode(string)
print(indices)
reconstructed_string = tokenizer.decode(indices)
print("decode:", reconstructed_string)
assert string == reconstructed_string
# 为了让模型能“认出”当前文本里的所有字符，它的词表至少要有多大。
vacabulary_size = max(indices) + 1
print(vacabulary_size)

# 评估分词器的效率。它衡量的是：将原始文本转换成数字序列后，数据是变“轻”了还是变“重”了？
# UTF-8 编码：这是一种变长编码。英文字符占 1 字节，常用中文字符占 3 字节，复杂的表情符号（如 🐋）占 4 字节。
def get_compression_ration(text, indices):
  import sys
  original_bytes = len(text.encode('utf-8'))
  # * 4 假设。在 Python 或深度学习框架（如 PyTorch）中，一个整数（int32）通常占用 4 字节。
  encoded_bytes = len(indices) * 4
  return original_bytes / encoded_bytes

compression_ratio = get_compression_ration(string, indices)
print(f"compression ratio: {compression_ratio}")

[104, 105, 65292, 24456, 22909, 30340, 65292, 116, 101, 114, 114, 105, 102, 105, 99, 65281, 128011]
decode: hi，很好的，terrific！🐋
128012
compression ratio: 0.47058823529411764


In [80]:
# 字节级Tokenizer
from collections import Counter
class ByteTokenizer:
 def __init__(self):
     self.vocab_size = 256

 def encode(self, text: str):
     return list(text.encode("utf-8"))

 def decode(self, indices):
     return bytes(indices).decode("utf-8")

# 字符级Tokenizer
class CharTokenizer:
 def __init__(self):
     self.vocab = {}
     self.inverse_vocab = {}

 def encode(self, text: str):
     tokens = []
     for ch in text:
         if ch not in self.vocab:
             idx = len(self.vocab)
             self.vocab[ch] = idx
             self.inverse_vocab[idx] = ch
         tokens.append(self.vocab[ch])
     return tokens

 def decode(self, indices):
     return "".join(self.inverse_vocab[i] for i in indices)

# 计算压缩率（byte/token）
def get_compression_ratio(text: str, token_len: int):
 input_byte_len = len(text.encode("utf-8"))
 return input_byte_len / token_len if token_len > 0 else 1


# 简易 BPE Tokenizer
class BPETokenizer:
 def __init__(self, num_merges):
     # num_merges: 训练的轮数。每一轮会产生一个新的 Token。这决定了最终词表的大小
     self.num_merges = num_merges
     # {(104, 105): 256} 表示把字符 'h' 和 'i' 合并为 ID 256
     self.merges = {}
     self.vocab_size = 256  # 从byte开始

 def get_stats(self, tokens):
     pairs = Counter()
     for i in range(len(tokens) - 1):
         pairs[(tokens[i], tokens[i+1])] += 1
     #print("-------", pairs)
     return pairs

 def merge_tokens(self, tokens, pair, new_token):
     i = 0
     new_tokens = []
     while i < len(tokens):
         if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
             new_tokens.append(new_token)
             i += 2
         else:
             new_tokens.append(tokens[i])
             i += 1
     return new_tokens


In [85]:
def run_tests():
    # Initialize Tokenizers
    byte_tok = ByteTokenizer()
    char_tok = CharTokenizer()
    bpe_tok = BPETokenizer(num_merges=10) # We will do 10 merges

    test_cases = [
        "banana banana",                   # Repetition
        "你好，世界！",                     # Chinese
        "AI is 💯 cool! 🚀",               # Mixed + Emoji
        "abababababababab"                 # Pure pattern for BPE
    ]

    for text in test_cases:
        print(f"\n{'='*10} Testing: '{text}' {'='*10}")

        # 1. Byte Tokenizer
        b_indices = byte_tok.encode(text)
        b_ratio = get_compression_ratio(text, len(b_indices))
        print(f"[Byte] Tokens: {len(b_indices)} | Ratio: {b_ratio:.2f}")

        # 2. Char Tokenizer
        c_indices = char_tok.encode(text)
        c_ratio = get_compression_ratio(text, len(c_indices))
        print(f"[Char] Tokens: {len(c_indices)} | Ratio: {c_ratio:.2f}")

        # 3. BPE Tokenizer (Training on the single sentence for demo)
        # In reality, BPE is trained on a huge corpus first.
        current_tokens = byte_tok.encode(text)
        for i in range(bpe_tok.num_merges):
            stats = bpe_tok.get_stats(current_tokens)
            print("-------",current_tokens, stats)
            if not stats: break
            # key=stats.get: 这是一个比较规则。它告诉 max 函数：“不要比较键本身，去调用 stats.get(key)，比较它们对应的值（次数）。”
            top_pair = max(stats, key=stats.get)
            new_id = 256 + i
            current_tokens = bpe_tok.merge_tokens(current_tokens, top_pair, new_id)
            # print("!!!!--",current_tokens)
            #print(new_id)

        bpe_ratio = get_compression_ratio(text, len(current_tokens))
        print(f"[BPE ] Tokens: {len(current_tokens)} | Ratio: {bpe_ratio:.2f} (after 10 merges)")

run_tests()



========== Testing: 'banana banana' ==========
[Byte] Tokens: 13 | Ratio: 1.00
[Char] Tokens: 13 | Ratio: 1.00
------- [98, 97, 110, 97, 110, 97, 32, 98, 97, 110, 97, 110, 97] Counter({(97, 110): 4, (110, 97): 4, (98, 97): 2, (97, 32): 1, (32, 98): 1})
!!!!-- [98, 256, 256, 97, 32, 98, 256, 256, 97]
------- [98, 256, 256, 97, 32, 98, 256, 256, 97] Counter({(98, 256): 2, (256, 256): 2, (256, 97): 2, (97, 32): 1, (32, 98): 1})
!!!!-- [257, 256, 97, 32, 257, 256, 97]
------- [257, 256, 97, 32, 257, 256, 97] Counter({(257, 256): 2, (256, 97): 2, (97, 32): 1, (32, 257): 1})
!!!!-- [258, 97, 32, 258, 97]
------- [258, 97, 32, 258, 97] Counter({(258, 97): 2, (97, 32): 1, (32, 258): 1})
!!!!-- [259, 32, 259]
------- [259, 32, 259] Counter({(259, 32): 1, (32, 259): 1})
!!!!-- [260, 259]
------- [260, 259] Counter({(260, 259): 1})
!!!!-- [261]
------- [261] Counter()
[BPE ] Tokens: 1 | Ratio: 13.00 (after 10 merges)

========== Testing: '你好，世界！' ==========
[Byte] Tokens: 18 | Ratio: 1.00
[Char]

In [86]:
# 词级分词器
import regex
"""
 \p{L}：代表任何语言的“字母”（Letter），包括中文、英文、俄文等。
\p{N}：代表任何形式的“数字”（Number）。
\p{S}：代表“符号”（Symbol），比如 Emoji 🐋
连续字母 (\p{L}+)
连续数字 (\p{N}+)
非字母数字的符号 ([^\p{L}\p{N}\s]+)
空格 (\s+)
"""
TOKENIZER_REGEX =  r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

def get_compression_ratio(text: str, segments):
  byte_len = len(text.encode("utf-8"))
  token_len = len(segments)
  return byte_len / token_len if token_len > 0 else 1

class WordTokenizer:
  # \w: 匹配任何 “字” (Word Character)。. (点): 在正则表达式中代表 “匹配除换行符以外的任何单个字符”。
  def __init__(self, pattern=r"\w+|.") -> None:
     self.pattern = pattern
     self.word2id = {}
     self.id2word = {}


  """
   text 是 "Hi, AI!"：
  扫描 "Hi"：符合 \w+（连续单词字符），被切下来。
  扫描 ","：不符合 \w+，但符合 .（任意字符），被单独切下来。
  扫描 " "：空格不符合 \w+，符合 .，被单独切下来。
  扫描 "AI"：符合 \w+，被切下来。
  扫描 "!"：不符合 \w+，符合 .，被单独切下来。
  最终结果 segments 为： ['Hi', ',', ' ', 'AI', '!']
  """
  def build_vocab(self, texts):
    vocab = set()
    for text in texts:
      segments = regex.findall(self.pattern, text)
      vocab.update(segments)

    vocab = sorted(vocab)
    self.word2id = {w: i for  i, w in enumerate(vocab)}
    self.id2word = {i: w for w, i in self.word2id.items() }

  def encode(self, text):
    """
    文本 → 字符串片段 → token id列表
    未登录词 UNK = -1
    参数 2 (-1)：默认值。如果字典里根本没见过这个词（即“未登录词”或 UNK），就返回 -1。
    """
    segments = regex.findall(self.pattern, text)
    return [self.word2id.get(seg, -1) for seg in segments], segments

  def decode(self, ids):
    """
    token ID → 原始片段 → 拼成字符串
    """
    return "".join(self.id2word.get(i, "<UNK>") for i in ids)


string = "It's so supercalifragilisticexpialidocious!👋👋"
print("原始字符串：", string)

# 使用基础正则分词（基于空格和标点切分）
basic_segments = regex.findall(r"\w+|.", string)
print("基础正则分词结果：")
print(basic_segments)

# 使用deepseek风格正则
segments = regex.findall(TOKENIZER_REGEX, string)
print(f"deepseek风格分词结果：{segments}")

# 构建词表
tokenizer = WordTokenizer(pattern=TOKENIZER_REGEX)
tokenizer.build_vocab([string])

print("词表大小：", len(tokenizer.word2id))

# 编码
ids, segs = tokenizer.encode(string)
print(f"编码token IDs：{ids}")

# 字节序列
byte_tokens = [b for b in string.encode("utf-8")]
print(f"UTF-8字节序列：{byte_tokens}")

print(f"编码segments：{segs}")

# 解码
decoded = tokenizer.decode(ids)
print("解码结果：", decoded)

# 压缩率
ratio = get_compression_ratio(string, segs)
print("压缩率：", ratio)

原始字符串： It's so supercalifragilisticexpialidocious!👋👋
基础正则分词结果：
['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!', '👋', '👋']
deepseek风格分词结果：['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!👋👋']
词表大小： 7
编码token IDs：[3, 2, 4, 0, 5, 0, 6, 1]
UTF-8字节序列：[73, 116, 39, 115, 32, 115, 111, 32, 115, 117, 112, 101, 114, 99, 97, 108, 105, 102, 114, 97, 103, 105, 108, 105, 115, 116, 105, 99, 101, 120, 112, 105, 97, 108, 105, 100, 111, 99, 105, 111, 117, 115, 33, 240, 159, 145, 139, 240, 159, 145, 139]
编码segments：['It', "'", 's', ' ', 'so', ' ', 'supercalifragilisticexpialidocious', '!👋👋']
解码结果： It's so supercalifragilisticexpialidocious!👋👋
压缩率： 6.375


In [107]:
import regex
from collections import Counter

# DeepSeek风格正则
DEEPSEEK_REGEX = r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

# 使用grapheme cluster保持emoji不被拆分
def split_graphemes(token):
    return tuple(regex.findall(r'\X', token))

# BPE训练函数
def train_bpe(texts, num_merges=50):
    """
    texts: 文本列表（用于训练BPE）
    num_merges: BPE 迭代合并的次数
    """
    # 1.构建初始vocab（字符级+</w>结束符）
    vocab = Counter()
    for text in texts:
        tokens = regex.findall(DEEPSEEK_REGEX, text)
        for token in tokens:
            chars = split_graphemes(token) + ('</w>',)
            vocab[chars] += 1
    print("0--", vocab)
    merges = []
    for _ in range(num_merges):
        # 统计相邻pair出现次数
        pairs = Counter()
        for word, freq in vocab.items():
            for i in range(len(word)-1):
                pairs[(word[i], word[i+1])] += freq
        if not pairs:
            break
        print("1--", pairs.items())
        # 找到最常见pair
        best_pair = max(pairs, key=pairs.get)
        merges.append(best_pair)
        print("2...",merges)

        # 合并所有vocab中的该pair
        new_vocab = {}
        for word, freq in vocab.items():
            w = []
            i = 0
            while i < len(word):
                if i < len(word)-1 and (word[i], word[i+1]) == best_pair:
                    w.append(word[i]+word[i+1])
                    i += 2
                else:
                    w.append(word[i])
                    i += 1
            new_vocab[tuple(w)] = freq
        vocab = new_vocab
    print("2--", vocab)
    print("3--",merges)
    return merges, vocab

# BPE Tokenizer类
class BPETokenizer:
    def __init__(self, merges):
        self.merges = merges

    def encode_word(self, token):
        # 初始分成字符+</w>
        word = list(split_graphemes(token)) + ['</w>']
        # 按merge顺序依次合并
        for pair in self.merges:
            i = 0
            new_word = []
            while i < len(word):
                if i < len(word)-1 and (word[i], word[i+1]) == pair:
                    new_word.append(word[i]+word[i+1])
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            word = new_word
        return word

    def encode(self, text):
        tokens = regex.findall(DEEPSEEK_REGEX, text)
        bpe_tokens = []
        for t in tokens:
            bpe_tokens.extend(self.encode_word(t))
        return bpe_tokens

    def decode(self, tokens):
        # 拼接tokens并去掉结尾</w>
        text = ''.join(tokens).replace('</w>', '')
        return text





In [108]:
train_texts = ["这只猫🐈很可爱", "the quick brown fox jumps over the lazy 🐕‍🦺"]
merges, vocab = train_bpe(train_texts, num_merges=20)
print("BPE合并:", merges)
tokenizer = BPETokenizer(merges)
test_text = "敏捷的棕色狐狸🦊"
encoded = tokenizer.encode(test_text)
print("编码:", encoded)
decoded = tokenizer.decode(encoded)
print("解码:", decoded)

0-- Counter({(' ', '</w>'): 8, ('t', 'h', 'e', '</w>'): 2, ('这', '只', '猫', '</w>'): 1, ('🐈', '</w>'): 1, ('很', '可', '爱', '</w>'): 1, ('q', 'u', 'i', 'c', 'k', '</w>'): 1, ('b', 'r', 'o', 'w', 'n', '</w>'): 1, ('f', 'o', 'x', '</w>'): 1, ('j', 'u', 'm', 'p', 's', '</w>'): 1, ('o', 'v', 'e', 'r', '</w>'): 1, ('l', 'a', 'z', 'y', '</w>'): 1, ('🐕\u200d🦺', '</w>'): 1})
1-- dict_items([(('这', '只'), 1), (('只', '猫'), 1), (('猫', '</w>'), 1), (('🐈', '</w>'), 1), (('很', '可'), 1), (('可', '爱'), 1), (('爱', '</w>'), 1), (('t', 'h'), 2), (('h', 'e'), 2), (('e', '</w>'), 2), ((' ', '</w>'), 8), (('q', 'u'), 1), (('u', 'i'), 1), (('i', 'c'), 1), (('c', 'k'), 1), (('k', '</w>'), 1), (('b', 'r'), 1), (('r', 'o'), 1), (('o', 'w'), 1), (('w', 'n'), 1), (('n', '</w>'), 1), (('f', 'o'), 1), (('o', 'x'), 1), (('x', '</w>'), 1), (('j', 'u'), 1), (('u', 'm'), 1), (('m', 'p'), 1), (('p', 's'), 1), (('s', '</w>'), 1), (('o', 'v'), 1), (('v', 'e'), 1), (('e', 'r'), 1), (('r', '</w>'), 1), (('l', 'a'), 1), (('a', 'z

In [109]:
import regex as re
from collections import Counter
from typing import List, Tuple, Dict, Iterable
import json
import base64

In [124]:
DEEPSEEK_REGEX = r"\p{L}+|\p{N}+|[^\p{L}\p{N}\s]+|\s+"

def pretokenize(text:str):
  return re.findall(DEEPSEEK_REGEX, text)


def bytes2tokens(b:bytes):
    """
    将UTF-8字节序列转为latin1可表示的token列表。
    每个字节0–255都能被latin1接映射到字符。
    """
    return [bytes([x]).decode('latin1') for x in b]

def tokens2bytes(tokens):
    """将latin1 token列表重新转回原始bytes"""
    return b''.join([t.encode('latin1') for t in tokens])

In [135]:
original_text = "Hi，中文, this is tesst！ 解码回原始文本🚀"

# 第一步：编码为 UTF-8 字节
raw_bytes = original_text.encode("utf-8")

# 第二步：转为 latin1 tokens (类似 GPT-2 的处理方式)
tokens = bytes2tokens(raw_bytes)
print(f"生成的 Tokens 列表: {tokens}")

# 第三步：使用 tokens2bytes 还原
restored_bytes = tokens2bytes(tokens)

# 第四步：解码回原始文本
final_text = restored_bytes.decode("utf-8")

print(f"还原后的文本: {final_text}")
print(f"数据是否完全一致: {original_text == final_text}")

生成的 Tokens 列表: ['H', 'i', 'ï', '¼', '\x8c', 'ä', '¸', '\xad', 'æ', '\x96', '\x87', ',', ' ', 't', 'h', 'i', 's', ' ', 'i', 's', ' ', 't', 'e', 's', 's', 't', 'ï', '¼', '\x81', ' ', 'è', '§', '£', 'ç', '\xa0', '\x81', 'å', '\x9b', '\x9e', 'å', '\x8e', '\x9f', 'å', '§', '\x8b', 'æ', '\x96', '\x87', 'æ', '\x9c', '¬', 'ð', '\x9f', '\x9a', '\x80']
还原后的文本: Hi，中文, this is tesst！ 解码回原始文本🚀
数据是否完全一致: True


In [165]:
def build_corpus(texts):
  corpus = []
  for text in texts:
    for chunk in pretokenize(text):
      corpus.append(bytes2tokens(chunk.encode('utf-8')))
  return corpus

def pair_freq(corpus: List[List[str]]):
  pairs = Counter()
  for word in corpus:
    #print(word)
    for i in range(len(word) - 1):
      pairs[(word[i], word[i+1])] +=1
  #print(pairs.items())
  return pairs

def merge_pair(word: List[str], pair: Tuple[str, str]):
  a, b = pair
  merge = []
  i = 0
  while i < len(word):
    if i < len(word) - 1 and word[i] == a and word[i+1] == b:
      merge.append(a+b)
      i += 2
    else:
      merge.append(word[i])
      i += 1
  #print(merge)
  return merge


In [173]:
# decode('latin1'): From Computer to Human  It takes a raw number (byte) and looks up what character it represents in the latin1 code book.
# encode('latin1'): From Human to Computer
def train_bpe(texts: Iterable[str], vocab_size: int=5000, num_merges: int=None) -> Tuple[List[Tuple[str,str]], List[str]]:
  corpus = build_corpus(texts)
  base_tokens = [bytes([i]).decode('latin1') for i in range(256)]
  merges: List[Tuple[str, str]] = []
  merged_set = set()
  cur_vocab_size = 256

  merge_steps = num_merges or (vocab_size - 256)

  for _ in range(merge_steps):
    pfreq = pair_freq(corpus)
    if not pfreq:
      break

    best_pair, _ = pfreq.most_common(1)[0]
    if cur_vocab_size + 1 > vocab_size:
      break

    merges.append(best_pair)

    corpus = [merge_pair(word, best_pair) for word in corpus]

    merged_set.add(best_pair)
    cur_vocab_size += 1

  special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
  vocab_tokens = special_tokens + base_tokens + sorted(merged_set)
  #print(vocab_tokens)
  return merges, vocab_tokens



# Tokenizer类
class DeepSeekV3Tokenizer:
    def __init__(self, merges: List[Tuple[str,str]], vocab_tokens: List[str]):
        self.merges = merges
        self.vocab_tokens = vocab_tokens

        # token ↔ id映射
        self.token2id = {tok:i for i, tok in enumerate(vocab_tokens)}
        self.id2token = {i:tok for tok,i in self.token2id.items()}

        # merges pair → 排序index
        self.ranks = {pair:i for i,pair in enumerate(merges)}

        # 特殊token
        self.pad_token = "<pad>"
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"

    def encode_chunk(self, chunk: str) -> List[str]:
        """
        对一个预分词做BPE编码：
        - 转字节token
        - 逐步应用merges
        - 处理OOV：未知token拆回字节或标记为<unk>
        """
        tokens = bytes2tokens(chunk.encode('utf-8'))

        # 应用PE并规则
        for pair in self.merges:
            new_tokens = []
            i = 0
            a,b = pair
            while i < len(tokens):
                if i<len(tokens)-1 and tokens[i]==a and tokens[i+1]==b:
                    new_tokens.append(a+b)
                    i+=2
                else:
                    new_tokens.append(tokens[i])
                    i+=1
            tokens = new_tokens

        # OOV token拆回字节
        out = []
        for t in tokens:
            if t in self.token2id:
                out.append(t)
            else:
                # 拆分成字节token，如果字节token也不在词表 → <unk>
                out.extend([ch if ch in self.token2id else self.unk_token for ch in t])
        return out

    def encode(self, text: str, add_bos=False, add_eos=False, print_chunks=False):
        """
        编码完整文本：
        - 先预分词
        - 再逐chunk编码
        - 可选打印中间过程
        """
        ids = []

        if add_bos:
            ids.append(self.token2id[self.bos_token])
            if print_chunks: print(f"[Special] <bos> -> {self.token2id[self.bos_token]}")

        for chunk in pretokenize(text):
            toks = self.encode_chunk(chunk)
            chunk_ids = [self.token2id.get(t, self.token2id[self.unk_token]) for t in toks]

            if print_chunks:
                readable = []
                for t in toks:
                    try:
                        # 尝试恢复utf-8
                        r = tokens2bytes([t]).decode('utf-8', errors='ignore')
                        readable.append(r if r else t.encode('latin1').hex())
                    except:
                        readable.append(t.encode('latin1').hex())

                print(f"[Chunk] \"{chunk}\" -> {readable} -> IDs: {chunk_ids}")

            ids.extend(chunk_ids)

        if add_eos:
            ids.append(self.token2id[self.eos_token])
            if print_chunks: print(f"[Special] <eos> -> {self.token2id[self.eos_token]}")
        return ids

    def decode(self, ids: Iterable[int]):
        """
        将ID序列还原为utf-8文本：
        """
        byte_seq = bytearray()
        for i in ids:
            tok = self.id2token.get(i, self.unk_token)
            if tok in {self.pad_token, self.bos_token, self.eos_token}:
                continue
            byte_seq.extend(tokens2bytes(list(tok)))
        # 如果你手动删掉了一些 ID 导致字节序列不完整，它会用 `` 替代而不是直接让程序崩溃。
        return byte_seq.decode('utf-8', errors='replace')

    def save(self, vocab_path: str, merges_path: str):

        # 保存vocab（token2id）
        with open(vocab_path, 'w', encoding='utf-8') as f:
            json.dump(self.token2id, f, ensure_ascii=False, indent=2)

        # 保存merges：每个token用base64
        merges_b64 = []
        for a, b in self.merges:
            a_bytes = a.encode('latin1')
            b_bytes = b.encode('latin1')
            merges_b64.append((
                base64.b64encode(a_bytes).decode('ascii'),
                base64.b64encode(b_bytes).decode('ascii')
            ))

        with open(merges_path, 'w', encoding='utf-8') as f:
            json.dump(merges_b64, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, vocab_path: str, merges_path: str):

        # 加载vocab
        with open(vocab_path, 'r', encoding='utf-8') as f:
            token2id = json.load(f)
        vocab_tokens = [None] * (max(token2id.values()) + 1)
        for tok, idx in token2id.items():
            vocab_tokens[idx] = tok

        # 加载merges（base64 → bytes → latin1）
        with open(merges_path, 'r', encoding='utf-8') as f:
            merges_b64 = json.load(f)

        merges = []
        for a_b64, b_b64 in merges_b64:
            a = base64.b64decode(a_b64).decode('latin1')
            b = base64.b64decode(b_b64).decode('latin1')
            merges.append((a, b))
        return cls(merges, vocab_tokens)



def train_tokenizer(texts, vocab_size=5000, num_merges=None):
    merges, vocab_tokens = train_bpe(texts, vocab_size=vocab_size, num_merges=num_merges)
    return DeepSeekV3Tokenizer(merges, vocab_tokens)


In [174]:
texts = [
    "Transformer是AI的核心技术。",
    "DeepSeek分词器支持中文、英文、emoji等多语言。",
    "Hello, 世界! 🌍🚀",
]

print("训练 Tokenizer (vocab_size=1024)")
tokenizer = train_tokenizer(texts, vocab_size=1024)
print(f"完成训练，词表大小: {len(tokenizer.vocab_tokens)}")
print("-"*50)

txt = "注意力机制是AI的核心技术。 🚀 🚀"
print(f"编码文本: {txt}")
ids = tokenizer.encode(txt, add_bos=True, add_eos=True, print_chunks=True)

print("-"*50)
print("Token ID:", ids)
decoded = tokenizer.decode(ids)
print("解码结果:", decoded)
print("是否可逆:", decoded == txt)

训练 Tokenizer (vocab_size=1024)
完成训练，词表大小: 352
--------------------------------------------------
编码文本: 注意力机制是AI的核心技术。 🚀 🚀
[Special] <bos> -> 1
[Chunk] "注意力机制是AI的核心技术" -> ['e6', 'b3', 'a8', 'e6', '84', '8f', 'e5', '8a', '9b', 'e6', '9c', 'ba', 'e5', '88', 'b6', 'e6', '98', 'af', 'A', 'I', 'e7', '9a', '84', 'e6', 'a0', 'b8', 'e5', 'bf', '83', 'e6', '8a', '80', 'e6', '9c', 'af'] -> IDs: [234, 183, 172, 234, 136, 147, 233, 142, 159, 234, 160, 190, 233, 140, 186, 234, 156, 179, 69, 77, 235, 158, 136, 234, 164, 188, 233, 195, 135, 234, 142, 132, 234, 160, 179]
[Chunk] "。" -> ['e3', '80', '82'] -> IDs: [231, 132, 134]
[Chunk] " " -> [' '] -> IDs: [36]
[Chunk] "🚀" -> ['f0', '9f', '9a', '80'] -> IDs: [244, 163, 158, 132]
[Chunk] " " -> [' '] -> IDs: [36]
[Chunk] "🚀" -> ['f0', '9f', '9a', '80'] -> IDs: [244, 163, 158, 132]
[Special] <eos> -> 2
--------------------------------------------------
Token ID: [1, 234, 183, 172, 234, 136, 147, 233, 142, 159, 234, 160, 190, 233, 140, 186, 234, 156, 179,


------------------------------
## 一、 核心设计思想 (Core Design)

   1. 字节为王 (Byte-level)：
   * 它不直接切分字符，而是先将文本转为 UTF-8 字节。
      * 为了防止处理字节时出现乱码或非法编码错误，它利用 latin1 将 0-255 每个字节映射为唯一的字符。
   2. 两阶段切分：
   * Pre-tokenize (预分词)：先用正则表达式把句子切成“单词/数字/符号”块，防止 BPE 把不相关的词（如数字和字母）粘在一起。
      * BPE Encoding：对每个块内部进行迭代合并。
   3. 安全性：
   * 通过 decode 时的 errors='replace' 和 encode_chunk 里的拆回字节逻辑，确保即使遇到损坏的数据也不会崩溃。
   
------------------------------
## 二、 逻辑流程伪代码 (Pseudo-code)## 1. 初始化 (__init__)

输入: merges (合并规则列表), vocab_tokens (所有可用的 token 列表)
过程:
    1. 构建 双向索引: token2id (词 -> 数字) 和 id2token (数字 -> 词)
    2. 构建 优先级表 (ranks): 记录合并规则的先后顺序
    3. 预设 特殊符号: <pad>, <bos>, <eos>, <unk> 的位置

## 2. 块编码 (encode_chunk) —— 最核心的算法

输入: chunk (预分词得到的一个短字符串)
过程:
    1. 转换为字节: raw_bytes = chunk.encode('utf-8')
    2. 映射为字符: tokens = 将每个字节按 latin1 转化为单个字符列表
    3. 循环迭代 merges:
        对于规则列表中的每一对 (a, b):
            在当前的 tokens 列表中查找相邻的 a 和 b
            若找到，将其合并为 "ab"
            继续扫描直至没有可合并的 a,b
    4. 处理 OOV (词表外词汇):
        检查合并后的每个 token 是否在训练好的词表中
        若不在，将其拆解回原始的单字节字符
    返回: 处理后的 token 列表

## 3. 完整编码 (encode)

输入: text, add_bos (是否加开头), add_eos (是否加结尾)
过程:
    1. 初始化 ids 列表
    2. 若 add_bos 为真: 插入 <bos> 的 ID
    3. 预分词: 使用正则将 text 切分为 chunks
    4. 循环处理每个 chunk:
        tokens = 调用 encode_chunk(chunk)
        将 tokens 转换为对应的数字 IDs 并存入结果
    5. 若 add_eos 为真: 插入 <eos> 的 ID
    返回: 最终的 ID 序列
